# Notebook 07 (Part B, CORRECTED): Hybrid Anomaly Detection Framework - Image

**This version fixes an error in the original Part B.** The classical component of the
image hybrid (Isolation Forest) must use features extracted from the **Convolutional
Autoencoder encoder**, matching the approved methodology and matching how Table II's
original Isolation Forest (Encoder) results were generated in Notebook 02. The original
version of this notebook mistakenly extracted features from the Deep SVDD encoder instead,
which was inconsistent with the rest of the study.

Run this on your local Dell (budget similar time to before, since two deep models are now
trained per run instead of one - likely 30-40 minutes per run, several hours total) or on
Colab with GPU (a few minutes total).

In [1]:
import os, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                              precision_score, recall_score, confusion_matrix)
from scipy.stats import wilcoxon
import pickle, warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cpu


In [2]:
BASE = r'C:\Users\Administrator\Documents\kf7029-w25039720'
AUG_PATH = fr'{BASE}\datasets\k-pipelines\K-Pipelines-main\K-Pipelines-main\Augmented'
TRAIN_PATH = os.path.join(AUG_PATH, 'train')
VALID_PATH = os.path.join(AUG_PATH, 'valid')
TEST_PATH  = os.path.join(AUG_PATH, 'test')

IMAGE_SIZE = 128
BATCH_SIZE = 32
transform = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor()])

train_dataset = ImageFolder(TRAIN_PATH, transform=transform)
valid_dataset = ImageFolder(VALID_PATH, transform=transform)
test_dataset  = ImageFolder(TEST_PATH,  transform=transform)

normal_cls = train_dataset.class_to_idx['no_corrosion']
corrosion_cls = test_dataset.class_to_idx['corrosion']
normal_indices = [i for i,(_,l) in enumerate(train_dataset.samples) if l == normal_cls]
train_normal = Subset(train_dataset, normal_indices)

test_labels  = np.array([corrosion_cls == l for _, l in test_dataset.samples]).astype(int)
valid_labels = np.array([corrosion_cls == l for _, l in valid_dataset.samples]).astype(int)

print(f'Normal training images: {len(train_normal)}')
print(f'Test: {len(test_dataset)} | Valid: {len(valid_dataset)}')

Normal training images: 434
Test: 108 | Valid: 108


## Model Definitions - Deep SVDD (Notebook 03) and Conv Autoencoder (Notebook 02), exactly as originally used

In [3]:
class DeepSVDD(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),   nn.LeakyReLU(0.1), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1, bias=False),  nn.LeakyReLU(0.1), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1, bias=False), nn.LeakyReLU(0.1), nn.MaxPool2d(2),
        )
        self.fc = nn.Linear(128 * 16 * 16, embed_dim, bias=False)
    def forward(self, x):
        x = self.features(x)
        return self.fc(x.view(x.size(0), -1))

class PretrainNet(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = nn.Sequential(
            nn.Linear(128, 128 * 16 * 16, bias=False),
            nn.Unflatten(1, (128, 16, 16)),
            nn.ConvTranspose2d(128, 64, 2, stride=2, bias=False), nn.LeakyReLU(0.1),
            nn.ConvTranspose2d(64, 32, 2, stride=2, bias=False),  nn.LeakyReLU(0.1),
            nn.ConvTranspose2d(32, 3, 2, stride=2, bias=False),   nn.Sigmoid()
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

# Conv Autoencoder - EXACTLY matching Notebook 02 (note: no bias=False, uses ReLU not LeakyReLU)
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),   nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 2, stride=2), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 2, stride=2),  nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 2, stride=2),   nn.Sigmoid()
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))
    def encode(self, x):
        return self.encoder(x)

def min_max_fit(train_scores):
    return train_scores.min(), train_scores.max()
def min_max_apply(scores, lo, hi):
    return (scores - lo) / (hi - lo + 1e-12)

def threshold_unsupervised(scores, anomaly_ratio=0.5):
    return np.percentile(scores, 100 * (1 - anomaly_ratio))

def collect_metrics(scores_test, y_test, threshold_ratio=0.5):
    t = threshold_unsupervised(scores_test, threshold_ratio)
    preds = (scores_test >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds, labels=[0,1]).ravel()
    fpr = fp / (fp + tn) if (fp+tn) > 0 else 0.0
    return {'ROC-AUC': roc_auc_score(y_test, scores_test), 'PR-AUC': average_precision_score(y_test, scores_test),
            'F1': f1_score(y_test, preds, zero_division=0), 'Precision': precision_score(y_test, preds, zero_division=0),
            'Recall': recall_score(y_test, preds, zero_division=0), 'FPR': fpr}

def extract_features(loader, model, device):
    feats_list, labels_list = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            f = model.encode(images)
            f = f.view(f.size(0), -1).cpu().numpy()
            feats_list.append(f)
            labels_list.extend(labels.numpy())
    return np.vstack(feats_list), np.array(labels_list)

print('Model definitions ready. Conv Autoencoder will provide encoder features for Isolation Forest, matching Notebook 02.')

Model definitions ready. Conv Autoencoder will provide encoder features for Isolation Forest, matching Notebook 02.


## 10-Run Training Loop

For each seed: retrain Deep SVDD (30 pretrain + 50 SVDD epochs) AND retrain the Conv
Autoencoder (50 epochs, matching Notebook 02) from scratch. Isolation Forest is fit on the
Conv Autoencoder's encoder features, not Deep SVDD's, correcting the earlier error.

In [4]:
SEEDS = [42, 7, 13, 99, 21, 55, 77, 3, 88, 11]
model_names = ['Deep SVDD', 'Isolation Forest (CAE-Enc)', 'Hybrid - Average', 'Hybrid - Weighted', 'Hybrid - Max']
metrics_list = ['ROC-AUC','PR-AUC','F1','Precision','Recall','FPR']
image_hybrid_runs = {name: {m: [] for m in metrics_list} for name in model_names}
selected_weights = []

train_loader_base = DataLoader(train_normal, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

def get_svdd_scores(encoder, loader, center):
    encoder.eval()
    scores = []
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            emb = encoder(images)
            d = torch.sum((emb - center)**2, dim=1)
            scores.extend(d.cpu().numpy())
    return np.array(scores)

t_start = time.time()
for run_idx, seed in enumerate(SEEDS):
    torch.manual_seed(seed)
    np.random.seed(seed)
    run_t0 = time.time()
    print(f'\n--- Run {run_idx+1}/10 (seed={seed}) | elapsed: {(run_t0-t_start)/60:.1f} min ---')

    # --- Deep SVDD ---
    encoder = DeepSVDD(embed_dim=128).to(device)
    pretrain_model = PretrainNet(encoder).to(device)
    optim_pre = torch.optim.Adam(pretrain_model.parameters(), lr=1e-4, weight_decay=1e-5)
    crit = nn.MSELoss()
    for epoch in range(30):
        pretrain_model.train()
        for images, _ in train_loader_base:
            images = images.to(device)
            optim_pre.zero_grad()
            loss = crit(pretrain_model(images), images)
            loss.backward()
            optim_pre.step()

    center = torch.zeros(128).to(device)
    encoder.eval()
    with torch.no_grad():
        for images, _ in train_loader_base:
            images = images.to(device)
            center += encoder(images).sum(dim=0)
    center /= len(train_normal)
    center[torch.abs(center) < 0.01] = 0.01

    optim_svdd = torch.optim.Adam(encoder.parameters(), lr=1e-4, weight_decay=1e-5)
    for epoch in range(50):
        encoder.train()
        for images, _ in train_loader_base:
            images = images.to(device)
            optim_svdd.zero_grad()
            emb = encoder(images)
            loss = torch.mean(torch.sum((emb - center)**2, dim=1))
            loss.backward()
            optim_svdd.step()

    svdd_train = get_svdd_scores(encoder, train_loader_base, center)
    svdd_val = get_svdd_scores(encoder, valid_loader, center)
    svdd_test = get_svdd_scores(encoder, test_loader, center)

    # --- Conv Autoencoder (for Isolation Forest features - CORRECTED) ---
    cae = ConvAutoencoder().to(device)
    cae_optim = torch.optim.Adam(cae.parameters(), lr=1e-3)
    for epoch in range(50):
        cae.train()
        for images, _ in train_loader_base:
            images = images.to(device)
            cae_optim.zero_grad()
            loss = crit(cae(images), images)
            loss.backward()
            cae_optim.step()

    train_feats, _ = extract_features(train_loader_base, cae, device)
    valid_feats, _ = extract_features(valid_loader, cae, device)
    test_feats, _ = extract_features(test_loader, cae, device)

    iso = IsolationForest(n_estimators=100, contamination=0.5, random_state=seed)
    iso.fit(train_feats)
    iso_train = -iso.score_samples(train_feats)
    iso_val = -iso.score_samples(valid_feats)
    iso_test = -iso.score_samples(test_feats)

    # --- Normalize and fuse ---
    svdd_lo, svdd_hi = min_max_fit(svdd_train)
    iso_lo, iso_hi = min_max_fit(iso_train)
    svdd_val_n, svdd_test_n = min_max_apply(svdd_val, svdd_lo, svdd_hi), min_max_apply(svdd_test, svdd_lo, svdd_hi)
    iso_val_n, iso_test_n = min_max_apply(iso_val, iso_lo, iso_hi), min_max_apply(iso_test, iso_lo, iso_hi)

    fus_avg_test = (svdd_test_n + iso_test_n) / 2

    best_w, best_pr = 0.5, -1
    for w in np.arange(0, 1.05, 0.05):
        pr = average_precision_score(valid_labels, w*svdd_val_n + (1-w)*iso_val_n)
        if pr > best_pr:
            best_pr, best_w = pr, w
    selected_weights.append(best_w)
    fus_w_test = best_w*svdd_test_n + (1-best_w)*iso_test_n

    fus_max_test = np.maximum(svdd_test_n, iso_test_n)

    run_scores = {
        'Deep SVDD': svdd_test,
        'Isolation Forest (CAE-Enc)': iso_test,
        'Hybrid - Average': fus_avg_test,
        'Hybrid - Weighted': fus_w_test,
        'Hybrid - Max': fus_max_test,
    }
    for name, scores in run_scores.items():
        m = collect_metrics(scores, test_labels, threshold_ratio=0.5)
        for k, v in m.items():
            image_hybrid_runs[name][k].append(v)

    run_time = time.time() - run_t0
    print(f'  Deep SVDD ROC-AUC={image_hybrid_runs["Deep SVDD"]["ROC-AUC"][-1]:.4f} | IF(CAE) ROC-AUC={image_hybrid_runs["Isolation Forest (CAE-Enc)"]["ROC-AUC"][-1]:.4f} | Selected weight={best_w:.2f} | run took {run_time/60:.1f} min')

total_time = time.time() - t_start
print(f'\nAll 10 runs complete in {total_time/60:.1f} minutes. Mean selected weight: {np.mean(selected_weights):.2f}')


--- Run 1/10 (seed=42) | elapsed: 0.0 min ---
  Deep SVDD ROC-AUC=0.7112 | IF(CAE) ROC-AUC=0.5703 | Selected weight=0.85 | run took 39.7 min

--- Run 2/10 (seed=7) | elapsed: 39.7 min ---
  Deep SVDD ROC-AUC=0.6595 | IF(CAE) ROC-AUC=0.5288 | Selected weight=0.10 | run took 34.0 min

--- Run 3/10 (seed=13) | elapsed: 73.7 min ---
  Deep SVDD ROC-AUC=0.7150 | IF(CAE) ROC-AUC=0.6015 | Selected weight=0.70 | run took 34.2 min

--- Run 4/10 (seed=99) | elapsed: 107.9 min ---
  Deep SVDD ROC-AUC=0.7253 | IF(CAE) ROC-AUC=0.5888 | Selected weight=0.45 | run took 33.6 min

--- Run 5/10 (seed=21) | elapsed: 141.6 min ---
  Deep SVDD ROC-AUC=0.6523 | IF(CAE) ROC-AUC=0.5549 | Selected weight=0.80 | run took 33.5 min

--- Run 6/10 (seed=55) | elapsed: 175.1 min ---
  Deep SVDD ROC-AUC=0.7064 | IF(CAE) ROC-AUC=0.5429 | Selected weight=0.80 | run took 33.7 min

--- Run 7/10 (seed=77) | elapsed: 208.7 min ---
  Deep SVDD ROC-AUC=0.6770 | IF(CAE) ROC-AUC=0.6152 | Selected weight=0.05 | run took 33.9 m

In [5]:
print('=== IMAGE HYBRID RESULTS - CORRECTED (mean ± std, 10 runs) ===\n')
rows = []
for name in model_names:
    r = image_hybrid_runs[name]
    rows.append({'Model': name,
        'ROC-AUC': f"{np.mean(r['ROC-AUC']):.4f} ± {np.std(r['ROC-AUC']):.4f}",
        'PR-AUC':  f"{np.mean(r['PR-AUC']):.4f} ± {np.std(r['PR-AUC']):.4f}",
        'F1':      f"{np.mean(r['F1']):.4f} ± {np.std(r['F1']):.4f}",
        'FPR':     f"{np.mean(r['FPR']):.4f} ± {np.std(r['FPR']):.4f}"})
df_img_hybrid = pd.DataFrame(rows).set_index('Model')
print(df_img_hybrid.to_string())

with open(fr'{BASE}\results\image_hybrid_runs_corrected.pkl', 'wb') as f:
    pickle.dump(image_hybrid_runs, f)
df_img_hybrid.to_csv(fr'{BASE}\results\image_hybrid_summary_corrected.csv')
print('\nSaved: results/image_hybrid_runs_corrected.pkl and results/image_hybrid_summary_corrected.csv')

=== IMAGE HYBRID RESULTS - CORRECTED (mean ± std, 10 runs) ===

                                    ROC-AUC           PR-AUC               F1              FPR
Model                                                                                         
Deep SVDD                   0.6823 ± 0.0352  0.7384 ± 0.0366  0.6278 ± 0.0356  0.3722 ± 0.0356
Isolation Forest (CAE-Enc)  0.5680 ± 0.0254  0.6122 ± 0.0193  0.5556 ± 0.0341  0.4444 ± 0.0341
Hybrid - Average            0.6798 ± 0.0410  0.7396 ± 0.0363  0.6056 ± 0.0511  0.3944 ± 0.0511
Hybrid - Weighted           0.6756 ± 0.0633  0.7278 ± 0.0560  0.6222 ± 0.0440  0.3778 ± 0.0440
Hybrid - Max                0.6660 ± 0.0372  0.7296 ± 0.0354  0.5963 ± 0.0387  0.4037 ± 0.0387

Saved: results/image_hybrid_runs_corrected.pkl and results/image_hybrid_summary_corrected.csv


## Statistical Testing - Hybrid Variants vs. Deep SVDD, with effect sizes

In [6]:
def effect_size_r(w_stat, n):
    mu = n*(n+1)/4
    sigma = np.sqrt(n*(n+1)*(2*n+1)/24)
    z = (w_stat - mu) / sigma
    return round(abs(z) / np.sqrt(n), 3)

baseline = image_hybrid_runs['Deep SVDD']['PR-AUC']
bonferroni_alpha = 0.05 / 3

print('=== Wilcoxon signed-rank: Hybrid variant vs. Deep SVDD (baseline), PR-AUC ===\n')
for name in ['Hybrid - Average', 'Hybrid - Weighted', 'Hybrid - Max']:
    comparison = image_hybrid_runs[name]['PR-AUC']
    stat, p = wilcoxon(baseline, comparison)
    r = effect_size_r(stat, len(baseline))
    sig = 'SIGNIFICANT' if p < bonferroni_alpha else 'not significant'
    direction = 'better' if np.mean(comparison) > np.mean(baseline) else 'worse or equal'
    print(f'{name:22s} mean PR-AUC={np.mean(comparison):.4f} vs SVDD={np.mean(baseline):.4f} | raw p={p:.4f} | effect size r={r} | vs Bonferroni-corrected alpha={bonferroni_alpha:.4f} -> {sig} | hybrid is {direction}')

=== Wilcoxon signed-rank: Hybrid variant vs. Deep SVDD (baseline), PR-AUC ===

Hybrid - Average       mean PR-AUC=0.7396 vs SVDD=0.7384 | raw p=0.7695 | effect size r=0.113 | vs Bonferroni-corrected alpha=0.0167 -> not significant | hybrid is better
Hybrid - Weighted      mean PR-AUC=0.7278 vs SVDD=0.7384 | raw p=0.8457 | effect size r=0.081 | vs Bonferroni-corrected alpha=0.0167 -> not significant | hybrid is worse or equal
Hybrid - Max           mean PR-AUC=0.7296 vs SVDD=0.7384 | raw p=0.2754 | effect size r=0.371 | vs Bonferroni-corrected alpha=0.0167 -> not significant | hybrid is worse or equal
